# 03 - Gold Layer

This notebook creates the final publish datasets and analytical answers for the Sales Data Engineering Assessment.

Gold outputs are saved as Parquet-backed logical tables under:

```text
parquet_tables/gold/
```

Main outputs:

- `publish_product`
- `publish_orders`
- `answer_highest_revenue_color_by_year`
- `answer_avg_lead_time_by_category`
- `quality_report`

This version is refactored for local VSCode execution and avoids Spark operations that may fail in Windows local mode, such as using `.count()` on a quality report DataFrame created from a Python list.

In [17]:
import json
import os
import shutil
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from setup import get_spark_session
from config import (
    SILVER_LAYER,
    GOLD_LAYER,
    SILVER_TABLES,
    GOLD_TABLES,
    FAIL_ON_CRITICAL,
)
from utils import (
    read_table,
    save_table,
    show_title,
    table_path,
)

spark = get_spark_session("sales-assessment-gold")

## Load quality check functions

The quality functions are kept in `00_quality_checks.ipynb`.

If this cell fails with `ModuleNotFoundError: No module named 'nbformat'`, run:

```python
%pip install nbformat
```

Then restart the kernel and run the notebook again.

In [18]:
%run ./00_quality_checks.ipynb

## Read Silver tables

In [19]:
show_title("Gold Layer - Reading Silver tables")

silver_sales_order_detail = read_table(
    spark,
    SILVER_LAYER,
    SILVER_TABLES["sales_order_detail"],
)

silver_sales_order_header = read_table(
    spark,
    SILVER_LAYER,
    SILVER_TABLES["sales_order_header"],
)

silver_products = read_table(
    spark,
    SILVER_LAYER,
    SILVER_TABLES["products"],
)

silver_sales_order_detail.printSchema()
silver_sales_order_header.printSchema()
silver_products.printSchema()


Gold Layer - Reading Silver tables
root
 |-- SalesOrderID: long (nullable = true)
 |-- SalesOrderDetailID: long (nullable = true)
 |-- OrderQty: integer (nullable = true)
 |-- ProductID: long (nullable = true)
 |-- UnitPrice: decimal(18,4) (nullable = true)
 |-- UnitPriceDiscount: decimal(18,4) (nullable = true)

root
 |-- SalesOrderID: long (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- ShipDate: date (nullable = true)
 |-- CustomerID: long (nullable = true)
 |-- SalesPersonID: long (nullable = true)
 |-- Freight: decimal(18,4) (nullable = true)

root
 |-- ProductID: long (nullable = true)
 |-- ProductDesc: string (nullable = true)
 |-- ProductNumber: string (nullable = true)
 |-- Color: string (nullable = true)
 |-- ProductSubCategoryName: string (nullable = true)
 |-- ProductCategoryName: string (nullable = true)



## Create `publish_product`

Business rules:

- Replace null `Color` values with `N/A`.
- Fill null `ProductCategoryName` based on `ProductSubCategoryName`.

In [20]:
show_title("Gold Layer - Creating publish_product")

clothing_subcategories = ["Gloves", "Shorts", "Socks", "Tights", "Vests"]
accessories_subcategories = ["Locks", "Lights", "Headsets", "Helmets", "Pedals", "Pumps"]
components_subcategories = ["Wheels", "Saddles"]

gold_publish_product = (
    silver_products
    .withColumn(
        "Color",
        F.coalesce(F.col("Color"), F.lit("N/A"))
    )
    .withColumn(
        "ProductCategoryName",
        F.when(
            F.col("ProductCategoryName").isNull()
            & F.col("ProductSubCategoryName").isin(clothing_subcategories),
            F.lit("Clothing"),
        )
        .when(
            F.col("ProductCategoryName").isNull()
            & F.col("ProductSubCategoryName").isin(accessories_subcategories),
            F.lit("Accessories"),
        )
        .when(
            F.col("ProductCategoryName").isNull()
            & (
                F.col("ProductSubCategoryName").contains("Frames")
                | F.col("ProductSubCategoryName").isin(components_subcategories)
            ),
            F.lit("Components"),
        )
        .otherwise(F.col("ProductCategoryName"))
    )
)

save_table(
    gold_publish_product,
    GOLD_LAYER,
    GOLD_TABLES["publish_product"],
)

print(f"Saved: {table_path(GOLD_LAYER, GOLD_TABLES['publish_product'])}")

gold_publish_product.show(10, truncate=False)


Gold Layer - Creating publish_product
Saved: c:\Users\bruna.martins\Downloads\tech_assessment\parquet_tables\gold\publish_product
+---------+--------------------------+-------------+-----+----------------------+-------------------+
|ProductID|ProductDesc               |ProductNumber|Color|ProductSubCategoryName|ProductCategoryName|
+---------+--------------------------+-------------+-----+----------------------+-------------------+
|680      |HL Road Frame - Black, 58 |FR-R92B-58   |Black|Road Frames           |Components         |
|706      |HL Road Frame - Red, 58   |FR-R92R-58   |Red  |Road Frames           |Components         |
|707      |Sport-100 Helmet, Red     |HL-U509-R    |Red  |Helmets               |Accessories        |
|708      |Sport-100 Helmet, Black   |HL-U509      |Black|Helmets               |Accessories        |
|709      |Mountain Bike Socks, M    |SO-B909-M    |White|Socks                 |Clothing           |
|710      |Mountain Bike Socks, L    |SO-B909-L    |W

## Create `publish_orders`

Business rules:

- Join Sales Order Detail with Sales Order Header using `SalesOrderID`.
- Include all fields from Sales Order Detail.
- Include all fields from Sales Order Header except `SalesOrderID`.
- Rename `Freight` to `TotalOrderFreight`.
- Calculate `TotalLineExtendedPrice`.
- Calculate `LeadTimeInBusinessDays`, excluding Saturdays and Sundays.
- Keep a `HasNegativeDateInterval` flag for records where `ShipDate < OrderDate`.
- Set `LeadTimeInBusinessDays` to null for invalid negative date intervals.

In [21]:
show_title("Gold Layer - Creating publish_orders")

header_with_lead_time = (
    silver_sales_order_header
    .withColumn(
        "HasNegativeDateInterval",
        F.when(
            F.col("OrderDate").isNotNull()
            & F.col("ShipDate").isNotNull()
            & (F.col("ShipDate") < F.col("OrderDate")),
            F.lit(True),
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "LeadTimeInBusinessDays",
        F.when(
            F.col("OrderDate").isNull() | F.col("ShipDate").isNull(),
            F.lit(None).cast("int")
        )
        .when(
            F.col("HasNegativeDateInterval"),
            F.lit(None).cast("int")
        )
        .when(
            F.col("ShipDate") > F.col("OrderDate"),
            F.size(
                F.expr(
                    """
                    filter(
                        sequence(OrderDate, date_sub(ShipDate, 1)),
                        x -> dayofweek(x) not in (1, 7)
                    )
                    """
                )
            )
        )
        .otherwise(F.lit(0))
    )
)

detail_cols = [
    F.col(f"d.{column_name}").alias(column_name)
    for column_name in silver_sales_order_detail.columns
]

header_cols = [
    (
        F.col(f"h.{column_name}").alias("TotalOrderFreight")
        if column_name == "Freight"
        else F.col(f"h.{column_name}").alias(column_name)
    )
    for column_name in header_with_lead_time.columns
    if column_name != "SalesOrderID"
]

gold_publish_orders = (
    silver_sales_order_detail.alias("d")
    .join(
        header_with_lead_time.alias("h"),
        F.col("d.SalesOrderID") == F.col("h.SalesOrderID"),
        "inner",
    )
    .select(
        *detail_cols,
        *header_cols,
        (
            F.col("d.OrderQty")
            * (F.col("d.UnitPrice") - F.col("d.UnitPriceDiscount"))
        ).alias("TotalLineExtendedPrice")
    )
)

save_table(
    gold_publish_orders,
    GOLD_LAYER,
    GOLD_TABLES["publish_orders"],
)

print(f"Saved: {table_path(GOLD_LAYER, GOLD_TABLES['publish_orders'])}")

gold_publish_orders.show(10, truncate=False)


Gold Layer - Creating publish_orders
Saved: c:\Users\bruna.martins\Downloads\tech_assessment\parquet_tables\gold\publish_orders
+------------+------------------+--------+---------+---------+-----------------+----------+----------+----------+-------------+-----------------+-----------------------+----------------------+----------------------+
|SalesOrderID|SalesOrderDetailID|OrderQty|ProductID|UnitPrice|UnitPriceDiscount|OrderDate |ShipDate  |CustomerID|SalesPersonID|TotalOrderFreight|HasNegativeDateInterval|LeadTimeInBusinessDays|TotalLineExtendedPrice|
+------------+------------------+--------+---------+---------+-----------------+----------+----------+----------+-------------+-----------------+-----------------------+----------------------+----------------------+
|43659       |1                 |1       |776      |2024.9940|0.0000           |2021-05-31|2021-06-07|29825     |279          |616.0984         |false                  |5                     |2024.9940             |
|43659 

## Analytical questions

In [22]:
show_title("Gold Layer - Analytical Questions")

orders_with_product = (
    gold_publish_orders.alias("o")
    .join(
        gold_publish_product
        .select(
            "ProductID",
            "Color",
            "ProductCategoryName",
        )
        .alias("p"),
        F.col("o.ProductID") == F.col("p.ProductID"),
        "left",
    )
    .select(
        F.col("o.*"),
        F.col("p.Color"),
        F.col("p.ProductCategoryName"),
    )
)

revenue_by_color_year = (
    orders_with_product
    .groupBy(
        F.year("OrderDate").alias("Year"),
        "Color",
    )
    .agg(
        F.sum("TotalLineExtendedPrice").alias("Revenue")
    )
)

rank_color_year = (
    Window
    .partitionBy("Year")
    .orderBy(F.col("Revenue").desc())
)

answer_highest_revenue_color_by_year = (
    revenue_by_color_year
    .withColumn("rn", F.row_number().over(rank_color_year))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .orderBy("Year")
)

answer_avg_lead_time_by_category = (
    orders_with_product
    .groupBy("ProductCategoryName")
    .agg(
        F.avg("LeadTimeInBusinessDays").alias("AvgLeadTimeInBusinessDays")
    )
    .orderBy(F.col("AvgLeadTimeInBusinessDays").desc_nulls_last())
)

save_table(
    answer_highest_revenue_color_by_year,
    GOLD_LAYER,
    GOLD_TABLES["highest_revenue_color_by_year"],
)

save_table(
    answer_avg_lead_time_by_category,
    GOLD_LAYER,
    GOLD_TABLES["avg_lead_time_by_category"],
)

print(f"Saved: {table_path(GOLD_LAYER, GOLD_TABLES['highest_revenue_color_by_year'])}")
print(f"Saved: {table_path(GOLD_LAYER, GOLD_TABLES['avg_lead_time_by_category'])}")

print("Which color generated the highest revenue each year?")
answer_highest_revenue_color_by_year.show(truncate=False)

print("What is the average LeadTimeInBusinessDays by ProductCategoryName?")
answer_avg_lead_time_by_category.show(truncate=False)


Gold Layer - Analytical Questions
Saved: c:\Users\bruna.martins\Downloads\tech_assessment\parquet_tables\gold\answer_highest_revenue_color_by_year
Saved: c:\Users\bruna.martins\Downloads\tech_assessment\parquet_tables\gold\answer_avg_lead_time_by_category
Which color generated the highest revenue each year?
+----+------+-------------+
|Year|Color |Revenue      |
+----+------+-------------+
|2021|Red   |6019634.2022 |
|2022|Black |14005242.9752|
|2023|Black |15047694.3692|
|2024|Yellow|6368158.4789 |
+----+------+-------------+

What is the average LeadTimeInBusinessDays by ProductCategoryName?
+-------------------+-------------------------+
|ProductCategoryName|AvgLeadTimeInBusinessDays|
+-------------------+-------------------------+
|NULL               |5.010809231668127        |
|Accessories        |5.006528417818741        |
|Clothing           |5.0050672138699275       |
|Bikes              |5.004896845147306        |
|Components         |5.0032938844517          |
+-------------

## Gold quality checks

This version avoids calling `evaluate_quality_report(gold_quality_report)` because that function performs a Spark `.count()` on the generated quality report DataFrame.

Instead, critical failures are counted directly from the Python list `quality_results`.

In [23]:
show_title("Gold Layer - Quality Checks")

quality_results = []

quality_results.append(
    check_primary_key_unique(
        gold_publish_product,
        GOLD_LAYER,
        GOLD_TABLES["publish_product"],
        "ProductID",
    )
)

quality_results.append(
    check_primary_key_unique(
        gold_publish_orders,
        GOLD_LAYER,
        GOLD_TABLES["publish_orders"],
        "SalesOrderDetailID",
    )
)

quality_results.append(
    check_foreign_key_exists(
        gold_publish_orders,
        gold_publish_product,
        GOLD_LAYER,
        GOLD_TABLES["publish_orders"],
        "ProductID",
        "ProductID",
    )
)

for col_name in [
    "SalesOrderID",
    "SalesOrderDetailID",
    "ProductID",
    "OrderDate",
    "ShipDate",
    "TotalLineExtendedPrice",
]:
    quality_results.append(
        check_not_null(
            gold_publish_orders,
            GOLD_LAYER,
            GOLD_TABLES["publish_orders"],
            col_name,
        )
    )

invalid_null_lead_time_count = (
    gold_publish_orders
    .filter(
        F.col("LeadTimeInBusinessDays").isNull()
        & (~F.col("HasNegativeDateInterval"))
        & F.col("OrderDate").isNotNull()
        & F.col("ShipDate").isNotNull()
    )
    .count()
)

quality_results.append(
    quality_result(
        layer=GOLD_LAYER,
        table_name=GOLD_TABLES["publish_orders"],
        check_name="lead_time_null_only_for_invalid_dates",
        severity="CRITICAL",
        failed_count=invalid_null_lead_time_count,
        rule_description=(
            "LeadTimeInBusinessDays may be null only when ShipDate is earlier than OrderDate "
            "or when OrderDate/ShipDate is null."
        ),
    )
)

for col_name in [
    "LeadTimeInBusinessDays",
    "TotalLineExtendedPrice",
    "TotalOrderFreight",
]:
    quality_results.append(
        check_non_negative(
            gold_publish_orders,
            GOLD_LAYER,
            GOLD_TABLES["publish_orders"],
            col_name,
        )
    )

quality_results.append(
    check_negative_date_interval(
        gold_publish_orders,
        GOLD_LAYER,
        GOLD_TABLES["publish_orders"],
        "OrderDate",
        "ShipDate",
    )
)

negative_dates_count = quarantine_negative_dates(
    gold_publish_orders,
    GOLD_TABLES["publish_orders"],
    start_date_col="OrderDate",
    end_date_col="ShipDate",
)

print(f"Negative date records quarantined from Gold orders: {negative_dates_count}")

tmp_quality_path = "parquet_tables/_tmp/gold_quality_report_json"

if os.path.exists(tmp_quality_path):
    shutil.rmtree(tmp_quality_path)

os.makedirs(tmp_quality_path, exist_ok=True)

json_file_path = os.path.join(tmp_quality_path, "part-00000.json")

with open(json_file_path, "w", encoding="utf-8") as file:
    for result in quality_results:
        file.write(json.dumps(result, default=str) + "\n")

gold_quality_report = spark.read.json(tmp_quality_path)

critical_failures = sum(
    1
    for result in quality_results
    if result["severity"] == "CRITICAL" and result["status"] == "FAIL"
)

save_table(
    gold_quality_report,
    GOLD_LAYER,
    GOLD_TABLES["quality_report"],
)

print(f"Saved: {table_path(GOLD_LAYER, GOLD_TABLES['quality_report'])}")

gold_quality_report.orderBy(
    "severity",
    "status",
    "table_name",
    "check_name",
).show(truncate=False)

print(f"Critical failures: {critical_failures}")

if FAIL_ON_CRITICAL and critical_failures > 0:
    raise Exception(
        f"Data quality failed with {critical_failures} critical failure(s)."
    )


Gold Layer - Quality Checks
Negative date records quarantined from Gold orders: 0
Saved: c:\Users\bruna.martins\Downloads\tech_assessment\parquet_tables\gold\quality_report
+-------------------------------------------------+------------+-----+-------------------------------------------------------------------------------------------------------------------+--------+------+---------------+
|check_name                                       |failed_count|layer|rule_description                                                                                                   |severity|status|table_name     |
+-------------------------------------------------+------------+-----+-------------------------------------------------------------------------------------------------------------------+--------+------+---------------+
|foreign_key_exists__ProductID                    |0           |gold |Every ProductID must exist in the referenced table.                                                